# Proyecto 1 — Monitoreo transaccional: detectar lo que el orden revela
Integrantes: Iris Ayala, Anggie Quezada

Dataset: Sparkov (Credit Card Transactions Fraud Detection)

In [ ]:
import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datos import cargar_datos, split_temporal, construir_agregados, preparar_features_evento, construir_secuencias
from modelo import (entrenar_modelo_A, predecir_modelo_A, entrenar_modelo_B, predecir_modelo_B,
                     evaluar, prueba_permutacion, analisis_costo)

SEMILLA = 42
np.random.seed(SEMILLA)

## 1. Integridad de datos
Origen, tamaño, tasa de fraude, split temporal.

In [ ]:
df = cargar_datos('data/fraudTrain.csv')
print(f"Total transacciones: {len(df)}")
print(f"Tasa de fraude: {df['is_fraud'].mean():.4%}")
print(f"Tarjetas únicas: {df['cc_num'].nunique()}")

train_raw, val_raw, test_raw = split_temporal(df)

In [ ]:
# Features de evento (para secuencias) -- el diccionario de categorías se arma con train y se reutiliza
train_ev, cat2cod = preparar_features_evento(train_raw)
val_ev, _ = preparar_features_evento(val_raw, cat2cod=cat2cod)
test_ev, _ = preparar_features_evento(test_raw, cat2cod=cat2cod)

# Agregados (para modelo A)
train_agg = construir_agregados(train_ev)
val_agg = construir_agregados(val_ev)
test_agg = construir_agregados(test_ev)

COLUMNAS_AGREGADAS = ['monto_prom_24h', 'n_tx_ultima_hora', 'monto_max_dia', 'diversidad_comercio']

for d in (train_agg, val_agg, test_agg):
    d[COLUMNAS_AGREGADAS] = d[COLUMNAS_AGREGADAS].fillna(0)

## 2. Núcleo común: A vs B

In [ ]:
# --- Modelo A: línea base sin orden ---
X_train_A = train_agg[COLUMNAS_AGREGADAS].values
y_train_A = train_agg['is_fraud'].values
X_test_A = test_agg[COLUMNAS_AGREGADAS].values
y_test_A = test_agg['is_fraud'].values

modelo_A, scaler_A = entrenar_modelo_A(X_train_A, y_train_A)
scores_A = predecir_modelo_A(modelo_A, scaler_A, X_test_A)
metricas_A = evaluar(y_test_A, scores_A)
print('Modelo A:', metricas_A)

In [ ]:
# --- Modelo B: secuencial ---
LONGITUD_SECUENCIA = 10  # <- decisión a documentar en el README (probar 5/10/20 y justificar)

sec_train, y_seq_train, _ = construir_secuencias(train_ev, longitud=LONGITUD_SECUENCIA)
sec_val, y_seq_val, _ = construir_secuencias(val_ev, longitud=LONGITUD_SECUENCIA)
sec_test, y_seq_test, _ = construir_secuencias(test_ev, longitud=LONGITUD_SECUENCIA)

modelo_B = entrenar_modelo_B(sec_train, y_seq_train, sec_val, y_seq_val,
                              n_features=sec_train.shape[2])

scores_B = predecir_modelo_B(modelo_B, sec_test)
metricas_B = evaluar(y_seq_test, scores_B)
print('Modelo B:', metricas_B)